In [11]:
%matplotlib inline

import os
import sys
from sys import platform

sys.path.append('../../')

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

import tiktoken

import torch
import torch.nn as nn



%load_ext autoreload
%autoreload 2

from llm_from_scratch.CH4.gpt import GPTModel
from llm_from_scratch.CH2.text_data_set import create_dataloader_v1
from llm_from_scratch.CH5.loss import calc_loss_loader
from llm_from_scratch.CH5.utils import load_weights_into_gpt, generate, text_to_token_ids, token_ids_to_text

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
#import urllib.request
#import ssl
#ssl._create_default_https_context = ssl._create_unverified_context # fix URLError SSL:CERTIFICIATE_VARIFICATION_FAILED

In [3]:
#url=("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch05/01_main-chapter-code/gpt_download.py")
#filename=url.split('/')[-1]
#print(f"{filename=}")
#urllib.request.urlretrieve(url, filename)

In [4]:
from gpt_download import download_and_load_gpt2

main_output_dirpath='../../../../../../results'
settings, params=download_and_load_gpt2(model_size="124M", models_dir=f"{main_output_dirpath}/gpts")

File already exists and is up-to-date: ../../../../../../results/gpts/124M/checkpoint
File already exists and is up-to-date: ../../../../../../results/gpts/124M/encoder.json
File already exists and is up-to-date: ../../../../../../results/gpts/124M/hparams.json
File already exists and is up-to-date: ../../../../../../results/gpts/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: ../../../../../../results/gpts/124M/model.ckpt.index
File already exists and is up-to-date: ../../../../../../results/gpts/124M/model.ckpt.meta
File already exists and is up-to-date: ../../../../../../results/gpts/124M/vocab.bpe


In [5]:
print(f"Settings: {settings}")
print(f"Parameter dictionary keys: {params.keys()}")

Settings: {'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}
Parameter dictionary keys: dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])


In [6]:
GPT_CONFIG_124M={'vocab_size':50257,
                 'context_length':256,
                 'emb_dim':768,
                 'n_heads':12,
                 'n_layers':12,
                 'drop_rate':0.1,
                 'qkv_bias':False}


model_configs={'gpt2-small (124M)':{'emb_dim':768, 'n_layers':12, 'n_heads':12},
               'gpt2-medium (355M)':{'emb_dim':1024,'n_layers':24,'n_heads':16},
               'gpt2-large (774M)':{'emb_dim':1280, 'n_layers':36, 'n_heads':20},
               'gpt2-xl (1558M)':{'emb_dim':1600, 'n_layers':48, 'n_heads':25}}

model_name='gpt2-small (124M)'
NEW_CONFIG=GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
print(f"{NEW_CONFIG=}")
NEW_CONFIG.update({'context_length':1024, 'qkv_bias':True})
print(f"{NEW_CONFIG=}")

NEW_CONFIG={'vocab_size': 50257, 'context_length': 256, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': False}
NEW_CONFIG={'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': True}


In [7]:
if any(x in platform for x in ('linux','win32')): 
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
elif platform=='darwin':  
    device=torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"{device=}")

gpt=GPTModel(NEW_CONFIG)
load_weights_into_gpt(gpt, params)
gpt.eval()
gpt.to(device);

device=device(type='mps')


In [8]:
torch.manual_seed(123)
tokenizer=tiktoken.get_encoding('gpt2')
token_ids=generate(model=gpt, idx=text_to_token_ids("Every effort moves you", tokenizer).to(device), 
                   max_new_tokens=25, context_size=NEW_CONFIG['context_length'], temperature=1.5, top_k=50, eos_id=None)
print('output text: ', token_ids_to_text(token_ids, tokenizer))

output text:  Every effort moves you like it. Once at the surface it is an uphill struggle in every bit as exciting as learning how easy something is going to


In [15]:
filepath='../the-verdict.txt'
with open(filepath, 'r', encoding='utf-8') as f: text_data=f.read()
train_ratio=0.9
split_idx=int(train_ratio*len(text_data))
train_data=text_data[:split_idx]
val_data=text_data[split_idx:]
print(f"{split_idx=}, {len(train_data)=}, {len(val_data)=}")
train_loader=create_dataloader_v1(train_data, batch_size=2, max_length=GPT_CONFIG_124M['context_length'],
                                  stride=GPT_CONFIG_124M['context_length'], drop_last=True, shuffle=True, num_workers=0)
val_loader=create_dataloader_v1(val_data, batch_size=2, max_length=GPT_CONFIG_124M['context_length'], 
                                stride=GPT_CONFIG_124M['context_length'], drop_last=False, shuffle=False, num_workers=0)
avg_train_loss=calc_loss_loader(data_loader=train_loader, model=gpt, device=device, num_batches=None)
avg_val_loss=calc_loss_loader(data_loader=val_loader, model=gpt, device=device, num_batches=None)
print(f"{avg_train_loss=:.3f}, {avg_val_loss=:.3f}")

split_idx=18431, len(train_data)=18431, len(val_data)=2048
avg_train_loss=3.755, avg_val_loss=3.560
